In [1]:
import datetime
import pandas as pd
from polygon import RESTClient

In [ ]:
# API_KEY = ""
client = RESTClient(api_key=API_KEY)

In [3]:
cols = [
    "<TICKER>",
    "<PER>",
    "<DATE>",
    "<TIME>",
    "<OPEN>",
    "<HIGH>",
    "<LOW>",
    "<CLOSE>",
    "<VOL>",
    "<OPENINT>",
    "<VWAP>",
    "<TRANSACTIONS>",
]
# all_data_list = [ticker_col, per_col, date_col, time_col, open_col, high_col, low_col, close_col, volume_col, openint_col, vwap_col, transactions_col]
# ticker_list = ["META", "AAPL", "MSFT", "GOOGL", "FB", "NFLX", "AMZN"]
ticker_list = [
    "MMM",
    "AXP",
    "AMGN",
    "AMZN",
    "AAPL",
    "BA",
    "CAT",
    "CVX",
    "CSCO",
    "KO",
    "DIS",
    "GS",
    "HD",
    "HON",
    "IBM",
    "JNJ",
    "JPM",
    "MCD",
    "MRK",
    "MSFT",
    "NKE",
    "NVDA",
    "PG",
    "CRM",
    "SHW",
    "TRV",
    "UNH",
    "VZ",
    "V",
    "WMT",
]

In [4]:
# List Aggregates (Bars)
for config in [(1, "minute"), (5, "minute"), (1, "hour"), (1, "day")]:
    for ticker in ticker_list:
        # aggs = []
        date_col = []
        time_col = []
        open_col = []
        high_col = []
        low_col = []
        close_col = []
        volume_col = []
        vwap_col = []
        transactions_col = []
        for a in client.list_aggs(
            ticker=ticker,
            multiplier=config[0],
            timespan=config[1],
            from_="2019-01-01",
            to="2025-02-06",
            adjusted=True,
            limit=50000,
        ):
            # aggs.append(a)

            dt = datetime.datetime.fromtimestamp(
                a.timestamp / 1000.0, tz=datetime.timezone.utc
            )
            date_formated = (
                f"{str(dt.year)}{str(dt.month).zfill(2)}{str(dt.day).zfill(2)}"
            )
            date_col.append(date_formated)
            time_formated = f"{str(dt.hour).zfill(2)}{str(dt.minute).zfill(2)}{str(dt.second).zfill(2)}"
            time_col.append(time_formated)

            open_col.append(a.open)
            high_col.append(a.high)
            low_col.append(a.low)
            close_col.append(a.close)
            volume_col.append(a.volume)
            vwap_col.append(a.vwap)
            transactions_col.append(a.transactions)

        openint_col = [0 for i in range(len(open_col))]
        per_col = [
            60 if config[1] == "hour" else config[0] for i in range(len(open_col))
        ]
        ticker_col = [f"{ticker}.US" for i in range(len(open_col))]

        all_data_list = [
            ticker_col,
            per_col,
            date_col,
            time_col,
            open_col,
            high_col,
            low_col,
            close_col,
            volume_col,
            openint_col,
            vwap_col,
            transactions_col,
        ]
        data_dict = {
            key_col: key_data for key_col, key_data in zip(cols, all_data_list)
        }
        data_df = pd.DataFrame.from_dict(data_dict)
        data_df.to_csv(
            f"./data/stocks/{config[0]}_{config[1]}/{ticker.lower()}.us.txt",
            index=False,
        )
        # len(aggs)